In [ ]:
!git clone https://github.com/YPolina/Medicine.git
%cd ./Medicine/BELKA/training
!pip install -r ../requirements.txt
from google.colab import drive
drive.mount('/content/drive')

In [16]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from lightgbm import LGBMClassifier
import pickle 
import os
from tqdm import tqdm
import h5py

In [19]:
fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=True)

def compute_fps(smiles_batch):

    """
    Compute fingerprints for a batch of SMILES
    """
    results = []
    for smiles in smiles_batch:
        mol = Chem.MolFromSmiles(smiles.replace('[Dy]', '[H]'))
        if mol is None:
            results.append((np.zeros(1024, dtype=np.int8), {}))
        else:
            fp = fpg.GetCountFingerprint(mol)
            arr = np.zeros(1024, dtype=np.int8)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            results.append(arr)
    return np.array(results)

def train_model_for_protein(protein_data, protein_name, batch_size=100000, lgb_params=None, eval_set=None):
    """
    Train a LightGBM model for a protein using batched data

    Args:
        protein_data (pd.DataFrame): Data containing SMILES strings and labels
        protein_name (str): Name of the protein
        batch_size (int): Number of samples per batch
        lgb_params (dict): Parameters for the LightGBM classifier
        eval_set (pd.DataFrame): Evaluation set for early stopping

    Returns:
        LGBMClassifier: Trained LightGBM model with the best iteration
    """
    #Split data into batches
    smiles_batches = np.array_split(protein_data['molecule_smiles'].tolist(), -(-len(protein_data) // batch_size))
    label_batches = np.array_split(protein_data['binds'].tolist(), -(-len(protein_data) // batch_size))

    #Prepare evaluation set
    if eval_set is not None and not eval_set.empty:
        eval_features = compute_fps(eval_set['molecule_smiles'])
        eval_labels = eval_set['binds']
        eval_set_lgb = [(eval_features, eval_labels)]

        with h5py.File(f'../intermediates/embeddings/{protein_name}_lightgbm_val.h5', 'w') as f:
            f.create_dataset('fingerprints', data=eval_features)
            f.create_dataset('labels', data=eval_labels)
    else:
        eval_set_lgb = None

    #Initialize the LightGBM
    lgb_cls = LGBMClassifier(**lgb_params)

    #Train the model in batches
    for smiles_batch, label_batch in tqdm(zip(smiles_batches, label_batches), total=len(smiles_batches), desc=f"Training {protein_name}"):
        if len(smiles_batch) != len(label_batch):
            continue

        #Compute fingerprints for the batch
        X_batch = compute_fps(smiles_batch)
        y_batch = np.array(label_batch)

        #Train
        lgb_cls.fit(
            X_batch, 
            y_batch, 
            eval_set=eval_set_lgb, 
            eval_metric='auc', 
            init_model=lgb_cls.booster_ if hasattr(lgb_cls, "booster_") else None
        )

    best_iteration = lgb_cls.best_iteration_
    print(f"Best iteration: {best_iteration}")

    lgb_cls.set_params(n_estimators=best_iteration)

    return lgb_cls

def save_models(model, protein, save_dir="../checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    model_path = os.path.join(save_dir, f"{protein}_lightgbm.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"Model for {protein} saved at {model_path}")

def train_models_by_protein(batch_size=100000, save_dir="../checkpoints", lgb_params = None, eval_set = None):
    protein_names = ['sEH', 'BRD4', 'HSA']

    for protein in tqdm(protein_names, desc="Training models"):

        train_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_train.parquet')
        val_data = pd.read_parquet(f'../intermediates/train_data/{protein}/{protein}_val.parquet')

        model = train_model_for_protein(train_data, protein, batch_size, lgb_params, val_data)

        save_models(model, protein, save_dir)

    return "Training complete"


In [21]:
lgb_params = {
        'max_depth': 11,
        'bagging_fraction': 0.9,
        'learning_rate': 0.05,
        'colsample_bytree': 1,
        'colsample_bynode': 0.5,
        'lambda_l1': 1,
        'objective': 'binary',
        'lambda_l2': 1.5,
        'num_leaves': 490,
        'min_data_in_leaf': 50,
        'verbose': -1,
        'metric': 'average_precision',
        'device': 'cpu',
        'early_stopping_rounds': 2
    }

train_models_by_protein(lgb_params=lgb_params)

Training models:   0%|          | 0/3 [00:00<?, ?it/s]/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medic

Best iteration: 1125
Model for sEH saved at ../checkpoints/sEH_lightgbm.pkl


/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/ut

Best iteration: 1077
Model for BRD4 saved at ../checkpoints/BRD4_lightgbm.pkl


/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/sklearn/ut

Best iteration: 769
Model for HSA saved at ../checkpoints/HSA_lightgbm.pkl


'Training complete'